# A branch declares the reason it refuses with

A refusing branch carries its own reason, so the caller reads the answer, not a guess.

[Script](22_reason_in_branch.py)


## Imports


In [ ]:
"""A branch declares the reason it refuses with."""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.access_control import Allowed, Refused, Verdict
from aoa.action_machine.intents.access_decide import access_decide
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.model.base_state import BaseState
from aoa.action_machine.resources import Connections
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.tools_box import ToolsBox


## The rule

A refusing branch carries its own reason, so the caller reads the answer, not a guess.


In [ ]:


class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class OrderParams(BaseParams):
    """Identify the order to cancel."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the cancelled order."""

    order_id: str = Field(description="Cancelled order identifier")


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole)
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):
    """Cancel an order after checking access."""

    @access_decide("Refuse an order that has already been closed")
    async def cancel_order_access_decide(
        self,
        params: OrderParams,
        box: ToolsBox,
        connections: Connections,
    ) -> Verdict:
        """The branch that refuses carries its own reason."""
        if params.order_id.startswith("CLOSED-"):
            return Refused("ORDER_ALREADY_CLOSED")
        return Allowed()

    @summary_aspect("Cancel the order")
    async def cancel_summary(self, params: OrderParams, state: BaseState, box: ToolsBox, connections: Connections) -> OrderResult:
        """Return the cancellation result."""
        return OrderResult(order_id=params.order_id)


## Try it

Run the cells in order. The calls below are the same as in the script.


In [ ]:
machine = ActionProductMachine(cache_coordinator=None)
caller = Context(user=UserInfo(user_id="m-1", roles=(ManagerRole,)))

answer = await machine.check_access_decide(caller, CancelOrderAction, OrderParams(order_id="CLOSED-9"))
print("closed order:", answer.model_dump(mode="json"))
answer = await machine.check_access_decide(caller, CancelOrderAction, OrderParams(order_id="ord-1"))
print("open order:", answer.model_dump(mode="json"))


closed order: {'kind': 'refused', 'gate': 'ACCESS_DECIDE', 'reason': 'ORDER_ALREADY_CLOSED'}
open order: {'kind': 'allowed'}
